# Module 1: Build a Sovereign Evaluation Dataset

⏱ **Estimated time:** 15 minutes

In this notebook, you'll look at the evaluation dataset every later module scores against, check it's valid, and upload it to S3 along with the training set used in Module 3.

## What You'll Do

1. Look at where the source material comes from, and its licence
2. Inspect the fields that make up each evaluation item
3. Validate the 50 evaluation items
4. Load the 200-item training set
5. Upload both datasets to S3

---
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.  
SPDX-License-Identifier: MIT-0

## Step 0: Install the Notebook Dependencies

Run this first to install the required packages for the notebook.

In [ ]:
# Install the notebook's dependencies (requirements-notebooks.txt sits beside the notebook)
get_ipython().run_line_magic("pip", "install -q -r requirements-notebooks.txt")
print("✅ Dependencies installed.")

## Configuration

Sets up the variables and AWS resources used in this notebook.

In [ ]:
# Settings, AWS clients, and the dataset bucket from the stack outputs
import boto3
import json
import os
from collections import Counter

# Settings: the AWS region, and which dataset to load
REGION = os.environ.get("AWS_REGION", "us-east-1")
DOMAIN = "legal"
JURISDICTION = "au"
NUM_ITEMS = 50

def stack_outputs():
    """Every CloudFormation output in the region, by key."""
    # The CloudFormation stack records the names of the resources it created
    cf = boto3.client("cloudformation", region_name=REGION)
    return {o["OutputKey"]: o["OutputValue"]
            for page in cf.get_paginator("describe_stacks").paginate()
            for s in page["Stacks"] for o in s.get("Outputs", [])}

# Look up the S3 bucket that holds the datasets
outputs = stack_outputs()
DATASET_BUCKET = outputs["DatasetBucketName"]
s3 = boto3.client("s3", region_name=REGION)

def read_jsonl(key):
    """Read a JSONL file from the dataset bucket."""
    # JSONL has one JSON record per line, so parse each non-empty line separately
    body = s3.get_object(Bucket=DATASET_BUCKET, Key=key)["Body"].read().decode()
    return [json.loads(line) for line in body.splitlines() if line.strip()]

print(f"Domain: {DOMAIN} | Jurisdiction: {JURISDICTION}")
print(f"Target: {NUM_ITEMS} items")
print(f"Bucket: {DATASET_BUCKET}")

## Step 1: Define Your Sources

The source must be the definitive authority for your domain and jurisdiction.
The Australian material used here is published under CC BY 4.0, which allows
reuse with attribution.

| Source | URL | Licence |
|--------|-----|---------|
| Federal Register of Legislation | [legislation.gov.au](https://www.legislation.gov.au) | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| NSW Legislation | [legislation.nsw.gov.au](https://legislation.nsw.gov.au) | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |

Check the licence for every source you add.

## Step 2: Structure Evaluation Items

Each item has a question, a reference answer taken from the legislation, and a
category. A good question has one clear answer, cites a specific section, and is distinguishable from international law.

In [ ]:
# Load the 50 evaluation items and count them by category
eval_items = read_jsonl(f"{DOMAIN}/eval-{DOMAIN}-{JURISDICTION}.jsonl")

# Count how many questions there are in each area of law
cats = Counter(item["category"] for item in eval_items)

print(f"Loaded {len(eval_items)} items. Categories ({len(cats)}):")
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count}")

## Step 3: Inspect Sample Items

Each item has a `prompt` (the question), a `referenceResponse` (the correct
answer, citing legislation) and a `category`. Note that each answer cites a
specific section.

Below is a demonstration of what an evaluation item looks like for you to inspect.

In [ ]:
# Print three sample items
# Each item has a question (prompt), the correct answer (referenceResponse) and an area of law (category)
for item in eval_items[:3]:
    print(f"Category: {item['category']}")
    print(f"Q: {item['prompt'][:100]}...")
    print(f"A: {item['referenceResponse'][:150]}...\n")

## Step 4: Validate Dataset

Checks that every item has the required fields. 
> A production dataset also needs review by qualified domain experts.

In [ ]:
# Check every item has the required fields
# Flag any item that's missing one of the three fields
errors = [f"Item {i}: missing {field}"
          for i, item in enumerate(eval_items)
          for field in ("prompt", "referenceResponse", "category") if field not in item]

# Flag reference answers too short to quote the legislation properly
errors += [f"Item {i}: referenceResponse too short"
           for i, item in enumerate(eval_items) if len(item.get("referenceResponse", "")) < 30]

# Print the first few problems, or confirm everything passed
if errors:
    print(f"✗ {len(errors)} problem(s):")
    for e in errors[:5]:
        print(f"  {e}")
else:
    print(f"✓ All {len(eval_items)} items pass structural validation")
    print(f"✓ {len(cats)} categories covered")
    print("⚠️  Expert validation required before production use")

## Step 5: Load the Fine-Tuning Dataset

The 200 training items cover the same categories and legislation as the 50
evaluation items, with no overlap. That way, a better score after fine-tuning
can't come from memorised answers.

Unlike the evaluation set, which is an answer key for scoring a model, the
training set is a set of example conversations for changing one. Each item has a
`system` prompt and a `messages` list: a `user` question followed by the
`assistant` answer the model should learn to give. 

In [ ]:
# Load the 200 training items and show one
# These are separate questions from the evaluation set, stored as example conversations
training_items = read_jsonl(f"{DOMAIN}/train-{DOMAIN}-{JURISDICTION}.jsonl")

print(f"Loaded {len(training_items)} training items. Sample:")
print(json.dumps(training_items[0], indent=2)[:300] + "...")
print(f"\nEval items:     {len(eval_items)} (scored in Module 2, never used for training)")
print(f"Training items: {len(training_items)} (used in Module 3, never scored)")

## Step 6: Upload to S3

The evaluation set and training set are uploaded to S3 for use by the following modules.

In [ ]:
# Upload both datasets to S3
# Write each dataset back as JSONL (one item per line) so later modules can read it
for name, items in (("eval", eval_items), ("train", training_items)):
    key = f"{DOMAIN}/{name}-{DOMAIN}-{JURISDICTION}.jsonl"
    s3.put_object(Bucket=DATASET_BUCKET, Key=key, ContentType="application/jsonl",
                  Body="\n".join(json.dumps(item) for item in items).encode())
    print(f"✓ s3://{DATASET_BUCKET}/{key} ({len(items)} items)")

print("\nDatasets ready. Proceed to Module 2.")

## Adapting This for Your Own Jurisdiction

1. Change `DOMAIN` and `JURISDICTION` in the configuration cell
2. Replace the dataset with questions from your domain's authoritative sources
3. Adjust the system prompt in the training data for your domain

The rest of the modules run unchanged.

## Success Criteria

Before moving on, verify that you have:

- [ ] Run Step 0 and restarted the kernel
- [ ] Seen the Configuration cell print your dataset bucket name
- [ ] Seen `All 50 items pass structural validation` and `10 categories covered`
- [ ] Loaded the 200-item training set and seen it is a different format to the eval set
- [ ] Confirmed both `.jsonl` files uploaded to S3

## Key Takeaways

- An evaluation item needs a question, an answer traceable to a specific section,
  and a category.
- If the answer would be the same under US law, the item tells you nothing about
  your own jurisdiction.
- The same source material gives you an evaluation set and a training set.
- Keep the two separate, or a better score may just mean the model memorised the answers.

## Next Steps

Open **Module-2-Evaluation.ipynb** to score 14 foundation models against the dataset
you just uploaded, using two independent evaluation paths.